In [46]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda

In [47]:
model = ChatOllama(model="deepseek-v3.1:671b-cloud")

In [48]:
class Feedback(BaseModel):
    feedback: Literal["positive", "negative"] = Field(
        description="The feedback can be either positive or negative."
    )

In [49]:
parser = PydanticOutputParser(pydantic_object=Feedback)


In [50]:
prompt = PromptTemplate(
    input_variables=["feedback"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
    template=(
        "You are a sentiment analysis assistant.\n"
        "Classify the sentiment of the following text as positive or negative.\n\n"
        "Text:\n{feedback}\n\n"
        "Follow these instructions when formatting your output:\n"
        "{format_instructions}"
    ),
)

In [51]:
prompt

PromptTemplate(input_variables=['feedback'], input_types={}, partial_variables={'format_instructions': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"feedback": {"description": "The feedback can be either positive or negative.", "enum": ["positive", "negative"], "title": "Feedback", "type": "string"}}, "required": ["feedback"]}\n```'}, template='You are a sentiment analysis assistant.\nClassify the sentiment of the following text as positive or negative.\n\nText:\n{feedback}\n\nFollow these instructions when formatting your output:\n{format_instructions}')

In [52]:
chain  = prompt | model | parser

In [53]:
chain.invoke({"feedback": "This product is amazing! I love it."})

Feedback(feedback='positive')

In [ ]:
prompt1 = PromptTemplate(
    input_variables=["feedback"],
    template=(
        "You are a helpful assistant that provides suggestions to improve the product based on customer feedback.\

In [ ]:
branch_chain = RunnableBranch(
    (
        lambda x: x["feedback"] == "positive",
        prompt | model | parser,
    ),
    (
        lambda x: x["feedback"] == "negative",
        prompt1 | model | parser,
    ),
)

TypeError: RunnableBranch default must be Runnable, callable or mapping.